# Ingest and Vectorize Zoning Laws

### Tech:

To extract raw text from pdfs use PyPDF2 or PyMuPDF/ for docs: python-docx

Langchain to handle splitting/chunking logic

Pinecone used as the vector store

Embeddings from OpenAI or another service



### Notes:

Semantic Chunking is preferred over Recursive Character splitter (standard) with overlap of some tokens based on semantics. To do that, you need embedding API and then do the semantic chunking, RCS does it to some extent but not as well as semantic chunking.

### Read the orlando zoning files

In [1]:
import os
import pickle
import gradio as gr

from openai import OpenAI
from dotenv import load_dotenv
from docx import Document
from pinecone import Pinecone, ServerlessSpec
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

load_dotenv()
client = OpenAI()

openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_api_key = Pinecone(api_key=pinecone_api_key)

if not openai_api_key or not pinecone_api_key:
    raise ValueError("OPENAI_API_KEY or PINECONE_API_KEY not found in .env")

print("API keys loaded successfully")

API keys loaded successfully


In [2]:
orlando_zoning_laws_path = "../data/orlando-zoning-laws"
files = os.listdir(orlando_zoning_laws_path)
for f in files:
    print(f)

Chapter_14___PROPERTY_MAINTENANCE_CODE.docx
Chapter_58___ZONING_DISTRICTS_AND_USES.docx
Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx
Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx
Chapter_63___ENVIRONMENTAL_PROTECTION.docx
Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx
Chapter_66___DEFINITIONS.docx
Chapter_67___AFFORDABLE_HOUSING.docx
Orlando-Zoning-Chapter-Descriptions.xlsx
OrlandoFLCodeofOrdinancesEXPORT20251209.xlsx


In [3]:
# Get only .docx files
docx_files = [f for f in os.listdir(orlando_zoning_laws_path) if f.endswith('.docx')]

# Dictionary to store text from each document
documents = {}

for filename in docx_files:
    filepath = os.path.join(orlando_zoning_laws_path, filename)
    doc = Document(filepath)
    
    # Extract all paragraphs
    text = "\n".join([para.text for para in doc.paragraphs])
    documents[filename] = text
    
    print(f"Loaded: {filename} ({len(text)} characters)")

print(f"\nTotal documents loaded: {len(documents)}")

Loaded: Chapter_14___PROPERTY_MAINTENANCE_CODE.docx (66196 characters)
Loaded: Chapter_58___ZONING_DISTRICTS_AND_USES.docx (595020 characters)
Loaded: Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx (110424 characters)
Loaded: Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx (171587 characters)
Loaded: Chapter_63___ENVIRONMENTAL_PROTECTION.docx (98777 characters)
Loaded: Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx (445724 characters)
Loaded: Chapter_66___DEFINITIONS.docx (175223 characters)
Loaded: Chapter_67___AFFORDABLE_HOUSING.docx (58442 characters)

Total documents loaded: 8


### Split and Chunk the orlando zoning laws

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chunk all documents
all_chunks = []

for filename, text in documents.items():
    chunks = splitter.create_documents([text])
    for i, chunk in enumerate(chunks):
        chunk.metadata = {
            "source": filename,
            "chunk_id": i,
            "total_chunks": len(chunks),
            "char_count": len(chunk.page_content),
            "chapter": filename.split("___")[0].replace("_", " "),
        }
    all_chunks.extend(chunks)
    print(f"{filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

Chapter_14___PROPERTY_MAINTENANCE_CODE.docx: 107 chunks
Chapter_58___ZONING_DISTRICTS_AND_USES.docx: 907 chunks
Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx: 171 chunks
Chapter_61___ROADWAY_DESIGN_AND_ACCESS_MANAGEMENT.docx: 257 chunks
Chapter_63___ENVIRONMENTAL_PROTECTION.docx: 138 chunks
Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx: 673 chunks
Chapter_66___DEFINITIONS.docx: 232 chunks
Chapter_67___AFFORDABLE_HOUSING.docx: 89 chunks

Total chunks: 2574


In [6]:
# Check a sample chunk
print(all_chunks[2570].page_content[:500])
print("\n--- Metadata ---")
print(all_chunks[2570].metadata)

Sec. 67.701. Affordable Housing Trust Fund Administration.
The Affordable Housing Trust Fund shall be administered in accordance with the policies and procedures established in Section 193.1 or other applicable policy within the Policies and Procedures Manual for the City of Orlando, as may be amended from time to time. 
(Ord. No. 2023-23, § 1, 9-11-2023, Doc. #2309111201)

Sec. 67.702. Annual Budget Requirements.
In approving the City's annual budget, the City Council shall have the absolute di

--- Metadata ---
{'source': 'Chapter_67___AFFORDABLE_HOUSING.docx', 'chunk_id': 85, 'total_chunks': 89, 'char_count': 733, 'chapter': 'Chapter 67'}


### Generate Embeddings using OpenAI. Store the vectors in Pinecone and Local Copy. Retrieve context

In [7]:
# Declaring the OpenAI ambeddings

# Recommended - explicitly pass API key
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Test it
# test_vector = embeddings.embed_query("zoning regulations in Orlando")
# print(f"Vector dimensions: {len(test_vector)}")

In [ ]:
# Create the index locally in the cloud

# Create a serverless index (free tier)
# pc.create_index(
#     name="orlando-zoning-index",
#     dimension=1536,  # text-embedding-3-small outputs 1536 dims
#     metric="cosine",
#     spec=ServerlessSpec(cloud="aws", region="us-east-1")
# )

In [8]:
# Index connection
index = pinecone_api_key.Index("orlando-zoning-index")

In [ ]:
# Pinecone upsert

batch_size = 100
vectors_to_upsert = []

for i, chunk in enumerate(all_chunks):
    # Generate embedding
    embedding = embeddings.embed_query(chunk.page_content)
    
    vectors_to_upsert.append({
        "id": f"{chunk.metadata['source']}_{chunk.metadata['chunk_id']}",
        "values": embedding,
        "metadata": {
            "text": chunk.page_content,  # important for retrieval!
            "source": chunk.metadata["source"],
            "chunk_id": chunk.metadata["chunk_id"],
            "chapter": chunk.metadata["chapter"]
        }
    })
    
    # Upsert every 100 chunks
    if len(vectors_to_upsert) >= batch_size:
        index.upsert(vectors=vectors_to_upsert)
        print(f"Upserted {i + 1} / {len(all_chunks)} chunks...")
        vectors_to_upsert = []

# Upsert remaining
if vectors_to_upsert:
    index.upsert(vectors=vectors_to_upsert)

print(f"✓ Done! Upserted {len(all_chunks)} chunks")
print(index.describe_index_stats())

In [ ]:
# Local Backup

local_backup = []
batch_size = 100
all_ids = [f"{chunk.metadata['source']}_{chunk.metadata['chunk_id']}" for chunk in all_chunks]

for i in range(0, len(all_ids), batch_size):
    batch_ids = all_ids[i:i + batch_size]
    result = index.fetch(ids=batch_ids)
    
    for chunk_id in batch_ids:
        if chunk_id in result.vectors:
            # Find matching chunk
            chunk = next(c for c in all_chunks if f"{c.metadata['source']}_{c.metadata['chunk_id']}" == chunk_id)
            local_backup.append({
                "id": chunk_id,
                "text": chunk.page_content,
                "metadata": chunk.metadata,
                "embedding": result.vectors[chunk_id].values
            })
    
    print(f"Fetched {min(i + batch_size, len(all_ids))} / {len(all_ids)}")

with open("../data/embeddings_orlando_zoning_laws_backup.pkl", "wb") as f:
    pickle.dump(local_backup, f)

print(f"✓ Saved {len(local_backup)} chunks")

In [ ]:
# Test

def ask(question, top_k=3):
    # Embed the question
    query_vector = embeddings.embed_query(question)
    
    # Query Pinecone
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    
    # Print results
    print(f"Question: {question}\n")
    for i, match in enumerate(results['matches'], 1):
        print(f"--- Result {i} (Score: {match['score']:.4f}) ---")
        print(f"Source: {match['metadata'].get('source')}")
        print(f"Chapter: {match['metadata'].get('chapter')}")
        print(f"Text: {match['metadata'].get('text', '')[:300]}...")
        print()

In [9]:
# Try it
ask("What size tree requires a removal permit?")

#Expected: 10 inches dbh

#All the 3 chunks produced similar result

Question: What size tree requires a removal permit?

--- Result 1 (Score: 0.7092) ---
Source: Chapter_60___SUBDIVISION_AND_LANDSCAPING.docx
Chapter: Chapter 60
Text: Sec. 60.209. General Requirements.
(a)	Tree Removal Permit Required. Removal of non-exempt existing 10" dbh or larger trees shall be prohibited without first obtaining a Tree Removal Permit. 
(b)	Mitigation. Tree Removal Permits may be approved where site design modifications are not feasible (see C...

--- Result 2 (Score: 0.6808) ---
Source: Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx
Chapter: Chapter 65
Text: Sec. 65.640. Tree Removal Permit.
When a Tree Removal Permit is Required. Unless exempt by section 163.045, Florida Statutes, persons desirous of removing any tree with a diameter at breast height of 10" or larger from any real property shall make application to and on a form prescribed by the Parks...

--- Result 3 (Score: 0.6807) ---
Source: Chapter_65___OFFICERS__BOARDS_AND_PROCEDURES.docx
Chapter: Chapter

### LLM call

In [10]:
def rag_query(question, top_k=3):
    # 1. Retrieve relevant chunks
    query_vector = embeddings.embed_query(question)
    results = index.query(vector=query_vector, top_k=top_k, include_metadata=True)
    
    # 2. Build context from chunks
    context = "\n\n".join([match['metadata']['text'] for match in results['matches']])
    
    # 3. Ask LLM with context
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer based only on the provided context. If the answer isn't in the context, say so."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    
    return response.choices[0].message.content

# Test it
print(rag_query("What size tree requires a removal permit?"))

A removal permit is required for non-exempt existing trees that are 10 inches in diameter at breast height (dbh) or larger.


### Chat History

In [11]:
class RAGChatbot:
    def __init__(self):
        self.history = []
    
    def chat(self, question, top_k=3):
        # Retrieve context
        query_vector = embeddings.embed_query(question)
        results = index.query(vector=query_vector, top_k=top_k, include_metadata=True)
        context = "\n\n".join([match['metadata']['text'] for match in results['matches']])
        
        # Build messages with history
        messages = [{"role": "system", "content": f"""You are a helpful assistant for Orlando zoning laws.
        Answer based on the provided context. If unsure, say so.
        
        Context:
        {context}"""}
        ]
        
        # Add conversation history
        messages.extend(self.history)
        
        # Add current question
        messages.append({"role": "user", "content": question})
        
        # Get response
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        
        answer = response.choices[0].message.content
        
        # Save to history
        self.history.append({"role": "user", "content": question})
        self.history.append({"role": "assistant", "content": answer})
        
        return answer
    
    def clear_history(self):
        self.history = []

# Usage
bot = RAGChatbot()

print(bot.chat("What size tree requires a permit?"))
# "A tree with 10 inches dbh or larger..."

print(bot.chat("What happens if I remove one without a permit?"))
# Now it knows "one" refers to a tree from previous context

print(bot.chat("Are there any exceptions?"))
# Continues the conversation

bot.clear_history()  # Start fresh

A Tree Removal Permit is required for the removal of non-exempt existing trees that have a diameter at breast height (dbh) of 10 inches or larger.
Removing a historic or specimen tree without a permit is deemed an irreparable and irreversible violation. A fine of up to fifteen thousand dollars ($15,000.00) may be imposed for such removal. The Parks Official will consider factors such as the gravity of the violation, any actions taken by the violator to correct the violation, and any previous violations committed by the violator when determining the amount of the fine.
Yes, there are exceptions to the requirement for a permit for tree removal. Specifically, no tree removal permit is required for the following:

1. Trees that are located on a single-family or duplex residential property.
2. Trees that are dead, diseased, or determined to be a hazard by the City.
3. Trees that are located within a roadway right-of-way and are removed in accordance with the City’s policies.

However, it is

### Gradio

In [ ]:
custom_css = """
/* Main container */
.gradio-container {
    font-family: 'Inter', 'Segoe UI', sans-serif;
    max-width: 1400px !important;
    margin: auto;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    min-height: 100vh;
}

/* Header styling */
.header-container {
    text-align: center;
    padding: 20px;
    background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
    border-radius: 15px;
    margin-bottom: 20px;
    box-shadow: 0 10px 40px rgba(102, 126, 234, 0.3);
}

.header-container h1 {
    color: white !important;
    font-size: 2.5em !important;
    margin: 0 !important;
    text-shadow: 2px 2px 4px rgba(0,0,0,0.2);
}

.header-container p {
    color: rgba(255,255,255,0.9) !important;
    font-size: 1.1em;
    margin-top: 10px !important;
}

/* Chat container */
.chatbot {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 15px !important;
    backdrop-filter: blur(10px);
}

/* Input box */
.input-textbox textarea {
    background: rgba(255,255,255,0.1) !important;
    border: 2px solid rgba(102, 126, 234, 0.5) !important;
    border-radius: 12px !important;
    color: white !important;
    font-size: 16px !important;
    padding: 15px !important;
    transition: all 0.3s ease;
}

.input-textbox textarea:focus {
    border-color: #667eea !important;
    box-shadow: 0 0 20px rgba(102, 126, 234, 0.3) !important;
}

/* Buttons */
.primary-btn {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
    padding: 12px 25px !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 5px 20px rgba(102, 126, 234, 0.3) !important;
}

.primary-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 30px rgba(102, 126, 234, 0.4) !important;
}

.secondary-btn {
    background: rgba(255,255,255,0.1) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    border-radius: 10px !important;
    color: white !important;
    transition: all 0.3s ease !important;
}

.secondary-btn:hover {
    background: rgba(255,255,255,0.2) !important;
    transform: translateY(-1px) !important;
}

/* Settings panel */
.settings-panel {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 15px !important;
    padding: 20px !important;
    backdrop-filter: blur(10px);
}

/* Example buttons */
.example-btn {
    background: rgba(102, 126, 234, 0.2) !important;
    border: 1px solid rgba(102, 126, 234, 0.3) !important;
    border-radius: 8px !important;
    color: white !important;
    margin: 3px 0 !important;
    text-align: left !important;
    font-size: 0.85em !important;
    transition: all 0.3s ease !important;
}

.example-btn:hover {
    background: rgba(102, 126, 234, 0.4) !important;
    border-color: #667eea !important;
}

/* Hide footer */
footer {
    display: none !important;
}

/* Scrollbar */
::-webkit-scrollbar {
    width: 8px;
}

::-webkit-scrollbar-track {
    background: rgba(255,255,255,0.05);
}

::-webkit-scrollbar-thumb {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    border-radius: 4px;
}
"""

# Track conversation stats
conversation_stats = {"questions": 0, "sources_used": set()}

def respond(message, history, top_k, show_sources, model_choice):
    global conversation_stats
    
    # 1. Retrieve relevant chunks
    query_vector = embeddings.embed_query(message)
    results = index.query(vector=query_vector, top_k=top_k, include_metadata=True)
    
    # 2. Build context
    chunks = results['matches']
    context = "\n\n".join([match['metadata']['text'] for match in chunks])
    
    # Track stats
    conversation_stats["questions"] += 1
    for match in chunks:
        conversation_stats["sources_used"].add(match['metadata'].get('chapter', 'Unknown'))
    
    # 3. Build messages with history
    messages = [
        {"role": "system", "content": f"""You are an expert assistant for Orlando zoning laws and city ordinances.
        INSTRUCTIONS:
        - Answer based ONLY on the provided context
        - Be clear, concise, and professional
        - Use bullet points for lists
        - Cite section numbers (e.g., "Sec. 60.209") when available
        - If the answer isn't in the context, say "I couldn't find that specific information in the Orlando zoning documents. Try rephrasing your question."

        Context:
        {context}"""}
    ]
    
    # Handle history based on format
    for item in history:
        if isinstance(item, dict):
            messages.append({"role": item["role"], "content": item["content"]}) 
    messages.append({"role": "user", "content": message})
    
    # 4. Generate response
    response = client.chat.completions.create(
        model=model_choice,
        messages=messages,
        temperature=0.3
    )
    
    answer = response.choices[0].message.content
    
    # 5. Add sources if enabled
    if show_sources:
        answer += "\n\n---\n"
        answer += "📚 **Sources Used:**\n"
        for i, match in enumerate(chunks, 1):
            chapter = match['metadata'].get('chapter', 'N/A')
            score = match['score']
            score_bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
            answer += f"\n**[{i}]** {chapter}\n"
            answer += f"    `{score_bar}` {score:.0%} match\n"
    
    return answer

def get_stats():
    return f"""
    
    ### 📊 Session Stats
    # - **Questions Asked:** {conversation_stats['questions']}
    # - **Chapters Referenced:** {len(conversation_stats['sources_used'])}"""

def clear_chat():
    global conversation_stats
    conversation_stats = {"questions": 0, "sources_used": set()}
    return [], get_stats()

# Build interface
with gr.Blocks(css=custom_css, title="Orlando Zoning Assistant") as demo:
    
    # Header
    gr.HTML("""
        <div class="header-container">
            <h1>🏛️ Orlando Zoning Laws Assistant</h1>
            <p>AI-powered search through Orlando's zoning regulations, permits, and city ordinances</p>
        </div>
    """)
    
    with gr.Row():
        # Main chat area
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                height=550,
                show_label=False,
                #type="tuples"
            )
            
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="💬 Ask about zoning laws, permits, regulations...",
                    show_label=False,
                    scale=8,
                    container=False,
                    elem_classes=["input-textbox"]
                )
                submit_btn = gr.Button("Send ➤", variant="primary", scale=1, elem_classes=["primary-btn"])
        
        # Sidebar
        with gr.Column(scale=1):
            # Settings
            with gr.Group(elem_classes=["settings-panel"]):
                gr.Markdown("### ⚙️ Settings")
                
                model_choice = gr.Dropdown(
                    choices=["gpt-4o-mini", "gpt-4o", "gpt-3.5-turbo"],
                    value="gpt-4o-mini",
                    label="Model",
                    interactive=True
                )
                
                top_k = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=3,
                    step=1,
                    label="Sources to Search",
                    info="More sources = more context"
                )
                
                show_sources = gr.Checkbox(
                    value=True,
                    label="📚 Show Source Citations"
                )
            
            # Stats
            stats_display = gr.Markdown(get_stats())
            
            # Actions
            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear", elem_classes=["secondary-btn"])
            
            # Examples
            gr.Markdown("### 💡 Quick Questions")
            
            examples = [
                ("🌳", "What size tree requires a removal permit?"),
                ("🏠", "What is the Affordable Housing Trust Fund?"),
                ("💰", "What's the fine for removing a historic tree?"),
                ("🌿", "What are the landscaping requirements?"),
                ("📋", "What is a specimen tree?"),
                ("🏗️", "What are zoning modification procedures?"),
            ]
            
            example_buttons = []
            for emoji, question in examples:
                btn = gr.Button(f"{emoji} {question}", size="sm", elem_classes=["example-btn"])
                example_buttons.append((btn, question))
    
    # Footer info
    gr.HTML("""
        <div style="text-align: center; padding: 20px; color: rgba(255,255,255,0.5); font-size: 0.85em;">
            Built with 🔗 LangChain • 🌲 Pinecone • 🤖 OpenAI • 🎨 Gradio<br>
            Data: Orlando City Code Zoning Regulations
        </div>
    """)
    
    # Event handlers
    def user_submit(message, history, top_k, show_sources, model_choice):
        if not message.strip():
            return "", history, get_stats()
        bot_response = respond(message, history, top_k, show_sources, model_choice)
        history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": bot_response}]
        return "", history, get_stats()
    
    # Connect events
    msg.submit(user_submit, [msg, chatbot, top_k, show_sources, model_choice], [msg, chatbot, stats_display])
    submit_btn.click(user_submit, [msg, chatbot, top_k, show_sources, model_choice], [msg, chatbot, stats_display])
    clear_btn.click(clear_chat, outputs=[chatbot, stats_display])
    
    for btn, question in example_buttons:
        btn.click(lambda q=question: q, outputs=msg)

demo.launch(share=True)